# Ekspor paket data (tanpa teks asli)
**CPU biasa, <2 menit.** Membuat folder `paket_data/` berisi label, skor, dan indeks split **tanpa kolom teks**, lalu men-zip-nya untuk diunggah ke GitHub. Notebook ini hanya membaca file di Drive; tidak ada model dilatih.
Aturan keamanan: hanya kolom `ID_teks` dan kolom numerik yang disimpan. Notebook berhenti jika ada kolom teks yang lolos.

In [ ]:
import os, glob, json, re, shutil, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'; DATA4=f'{DRIVE}/JURNAL-DATASET'; OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'; PREP4=f'{OUT4}/prep'
PKG=f'{OUT4}/paket_data'; os.makedirs(f'{PKG}/data/labels',exist_ok=True); os.makedirs(f'{PKG}/data/splits',exist_ok=True)
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
def find(name):
    h=sorted(set(glob.glob(f'{DATA4}/**/{name}',recursive=True))); print(name,'->',h[:2]); return h[0] if h else None
def safe(d):
    keep=['ID_teks']+[c for c in d.columns if c!='ID_teks' and pd.api.types.is_numeric_dtype(d[c])]
    out=d[keep].copy(); out['ID_teks']=out['ID_teks'].astype(str)
    print('  disimpan:',keep,'| dibuang:',[c for c in d.columns if c not in keep]); return out

In [ ]:
SRC={'gold_1981_data_ahli':'data_ahli_FIXED.csv','natural_2000_pseudolabel':'anotasi_natural_2000.csv','gold_batch3_577':'gold_batch3_577.csv','gold_batch4_564':'gold_batch4_600.csv','gold_batch5_600':'gold_batch5_600.csv'}
L={}
for k,fn in SRC.items():
    p=find(fn)
    if not p: print('  !! TIDAK ADA',fn); continue
    d=pd.read_csv(p,encoding='utf-8-sig'); print(k,d.shape); L[k]=safe(d); L[k].to_csv(f'{PKG}/data/labels/{k}.csv',index=False)

In [ ]:
# Split beku: ID per split untuk 1.981 gold (indeks dari NB07)
ids=L['gold_1981_data_ahli']['ID_teks'].values
idx_tr=np.load(f'{PREP4}/idx_tr_all_v3.npy'); idx_va=np.load(f'{PREP4}/idx_va_v3.npy'); idx_ho=np.load(f'{PREP4}/idx_ho_v3.npy')
sp=pd.DataFrame({'ID_teks':ids,'split':'unassigned'}); sp.loc[idx_tr,'split']='train'; sp.loc[idx_va,'split']='val'; sp.loc[idx_ho,'split']='test'
print(sp.split.value_counts().to_dict())
assert len(set(idx_tr)&set(idx_ho))==0 and len(set(idx_tr)&set(idx_va))==0
sp.to_csv(f'{PKG}/data/splits/gold_split_v3.csv',index=False)
ho=np.load(f'{PREP4}/ho_ids.npy',allow_pickle=True).astype(str); pd.DataFrame({'ID_teks':ho}).to_csv(f'{PKG}/data/splits/test_271_order.csv',index=False)
t130=pd.read_csv(find('test_bersih_130.csv'),encoding='utf-8-sig')[['ID_teks']].astype(str); t130.to_csv(f'{PKG}/data/splits/test_consensus_130_ids.csv',index=False)
print('test urutan',len(ho),'| konsensus',len(t130))

In [ ]:
# Label valid 271 (konsensus + adjudikasi) tanpa teks
def find2(name):
    h = sorted(set(glob.glob(f'{DATA4}/**/{name}', recursive=True)
                   + glob.glob(f'{DRIVE}/tesis-dataset*/{name}')
                   + glob.glob(f'{DRIVE}/tesis-dataset*/**/{name}', recursive=True)))
    pref = [x for x in h if 'tesis-dataset 3' in x]   # sama dengan yang dipakai NB15
    h = pref or h
    print(name, '->', h[:3]); return h[0] if h else None

p_valid = find2('test_271_valid.csv')
assert p_valid, 'test_271_valid.csv tidak ditemukan - cari manual di Drive'
tv = safe(pd.read_csv(p_valid, encoding='utf-8-sig'))
tv.to_csv(f'{PKG}/data/labels/test_271_valid_adjudicated.csv', index=False)
t130b = safe(pd.read_csv(find2('test_bersih_130.csv'), encoding='utf-8-sig'))
t130b.to_csv(f'{PKG}/data/labels/test_consensus_130_labels.csv', index=False)
print(tv.shape, t130b.shape)

In [ ]:
# PEMERIKSAAN KEAMANAN: tidak boleh ada teks asli di paket
bad=[]
for f in glob.glob(f'{PKG}/**/*.csv',recursive=True):
    d=pd.read_csv(f)
    for c in d.columns:
        if d[c].dtype==object and c not in ('ID_teks','split'): bad.append((f,c))
        if d[c].dtype==object and d[c].astype(str).str.len().max()>40 and c!='ID_teks': bad.append((f,c,'panjang'))
print('kolom mencurigakan:',bad); assert not bad,'ADA KOLOM TEKS - JANGAN DIUNGGAH'
print('file:',sorted(os.path.relpath(f,PKG) for f in glob.glob(f"{PKG}/**/*.*",recursive=True)))

In [ ]:
shutil.make_archive('/content/paket_data_tanpa_teks','zip',PKG)
from google.colab import files; files.download('/content/paket_data_tanpa_teks.zip')